In [ ]:
import cv2
import numpy as np


def calculate_percentage(mask, total_pixels):
    pixels = cv2.countNonZero(mask)
    return (pixels / total_pixels) * 100


def analyze_plant(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    total_pixels = frame.shape[0] * frame.shape[1]

    # HSV ranges
    lower_green = np.array([35, 40, 40], dtype=np.uint8)
    upper_green = np.array([90, 255, 255], dtype=np.uint8)

    lower_yellow = np.array([15, 40, 40], dtype=np.uint8)
    upper_yellow = np.array([35, 255, 255], dtype=np.uint8)

    lower_brown = np.array([5, 50, 20], dtype=np.uint8)
    upper_brown = np.array([20, 255, 200], dtype=np.uint8)

    # Masks
    green_mask = cv2.inRange(hsv, lower_green, upper_green)
    yellow_mask = cv2.inRange(hsv, lower_yellow, upper_yellow)
    brown_mask = cv2.inRange(hsv, lower_brown, upper_brown)

    # Clean masks a bit
    kernel = np.ones((5, 5), np.uint8)
    green_mask = cv2.morphologyEx(green_mask, cv2.MORPH_OPEN, kernel)
    yellow_mask = cv2.morphologyEx(yellow_mask, cv2.MORPH_OPEN, kernel)
    brown_mask = cv2.morphologyEx(brown_mask, cv2.MORPH_OPEN, kernel)

    # Percentages
    green_pct = calculate_percentage(green_mask, total_pixels)
    yellow_pct = calculate_percentage(yellow_mask, total_pixels)
    brown_pct = calculate_percentage(brown_mask, total_pixels)

    # Leaf coverage = total visible plant-like area
    leaf_coverage = green_pct + yellow_pct + brown_pct

    # Dryness score: more weight to brown, some to yellow
    dryness_score = (brown_pct * 0.7) + (yellow_pct * 0.3)

    # Stress score: healthy green reduces stress, yellow/brown increase stress
    stress_score = max(0, min(100, (yellow_pct * 1.2) + (brown_pct * 1.8) - (green_pct * 0.5) + 30))

    # Health classification
    if green_pct >= 30 and yellow_pct < 10 and brown_pct < 5:
        health = "Healthy"
        stress = "Low Stress"
        color = (0, 255, 0)
    elif green_pct >= 20 and brown_pct < 15:
        health = "Moderate"
        stress = "Medium Stress"
        color = (0, 255, 255)
    else:
        health = "Unhealthy"
        stress = "High Stress"
        color = (0, 0, 255)

    return {
        "green_mask": green_mask,
        "yellow_mask": yellow_mask,
        "brown_mask": brown_mask,
        "green_pct": green_pct,
        "yellow_pct": yellow_pct,
        "brown_pct": brown_pct,
        "leaf_coverage": leaf_coverage,
        "dryness_score": dryness_score,
        "stress_score": stress_score,
        "health": health,
        "stress": stress,
        "color": color,
    }


def create_thermal_map(mask):
    blurred = cv2.GaussianBlur(mask, (21, 21), 0)
    thermal = cv2.applyColorMap(blurred, cv2.COLORMAP_JET)
    return thermal


def process_frame(frame, alpha=0.45):
    result = analyze_plant(frame)

    thermal_map = create_thermal_map(result["green_mask"])

    overlay = cv2.addWeighted(frame, 1 - alpha, thermal_map, alpha, 0)
    overlay[result["green_mask"] == 0] = frame[result["green_mask"] == 0]

    color = result["color"]

    cv2.putText(overlay, f"Health: {result['health']}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.putText(overlay, f"Stress: {result['stress']}", (20, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.putText(overlay, f"Green: {result['green_pct']:.2f}%", (20, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.putText(overlay, f"Yellow: {result['yellow_pct']:.2f}%", (20, 130),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.putText(overlay, f"Brown: {result['brown_pct']:.2f}%", (20, 160),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (42, 42, 165), 2)

    cv2.putText(overlay, f"Leaf Coverage: {result['leaf_coverage']:.2f}%", (20, 200),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(overlay, f"Dryness Score: {result['dryness_score']:.2f}", (20, 230),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 255), 2)

    cv2.putText(overlay, f"Stress Score: {result['stress_score']:.2f}", (20, 260),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 200, 200), 2)

    return result["green_mask"], result["yellow_mask"], result["brown_mask"], thermal_map, overlay


def run_video(source=0):
    cap = cv2.VideoCapture(source)

    if not cap.isOpened():
        print("Error: Could not open webcam/video source.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        green_mask, yellow_mask, brown_mask, thermal_map, overlay = process_frame(frame)

        cv2.imshow("Original Frame", frame)
        cv2.imshow("Green Mask", green_mask)
        cv2.imshow("Yellow Mask", yellow_mask)
        cv2.imshow("Brown Mask", brown_mask)
        cv2.imshow("Thermal Map", thermal_map)
        cv2.imshow("Plant Health and Stress Analysis", overlay)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    run_video(0)

